# 01c · Máquinas de vectores de soporte: el margen, la bisagra y el kernel

**Módulo 4 · Sesión 9** — KNN y SVM

## Objetivos

`02-knn-y-svm.md` plantea la SVM como un problema de optimización con restricciones. Este
notebook lo **dibuja**, con los mismos 20 estudiantes del notebook 01b:

1. Cuando dos clases son separables hay **infinitas rectas** que las separan; cada una
   deja una "calle" vacía a su alrededor, y la SVM elige la **más ancha**.
2. La geometría de una recta $\mathbf{w}^\top\mathbf{x} + b = 0$: por qué $\mathbf{w}$ es
   perpendicular a ella y por qué el ancho de la calle es $2/\lVert\mathbf{w}\rVert$.
3. El margen máximo resuelto a mano con `scipy`, validado contra `SVC`, y los **vectores
   de soporte**: los únicos puntos que importan.
4. El soft margin: la pérdida **bisagra** (*hinge*) frente a la entropía cruzada de la
   logística, y $C$ como el precio de invadir la calle.
5. El **truco del kernel**: separar con un plano en un espacio más grande, sin construirlo;
   y la SVM con kernel RBF como una suma de "montañas" gaussianas centradas en los vectores
   de soporte.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy.optimize import minimize
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Muchas rectas separan a los mismos 20 estudiantes

Los 20 estudiantes del notebook 01b, con las variables estandarizadas. La SVM trabaja con
etiquetas $y_i \in \{-1, +1\}$ en lugar de $\{0, 1\}$: así "estar del lado correcto" se
escribe en una sola expresión, $y_i f(\mathbf{x}_i) > 0$, para las dos clases.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
VARIABLES = ["promedio_anterior", "horas_estudio_semana"]

muestra = datos.sample(n=20, random_state=SEMILLA)
X_crudo = muestra[VARIABLES].to_numpy()
X = (X_crudo - X_crudo.mean(axis=0)) / X_crudo.std(axis=0)
y = 2 * muestra["aprobo"].to_numpy() - 1  # {0, 1} -> {-1, +1}

COLORES = {-1: "tab:red", 1: "tab:blue"}
FONDO = ListedColormap(["#f4c7c3", "#c6dbef"])
LIM = (-2.6, 2.6)


def preparar_eje(eje, titulo="", lim=LIM):
    eje.set_xlim(*lim)
    eje.set_ylim(*lim)
    eje.set_aspect("equal")
    eje.set_xlabel("promedio_anterior (estandarizado)")
    eje.set_ylabel("horas_estudio (estandarizado)")
    eje.set_title(titulo)


def dibujar_puntos(eje, X, y, s=45):
    for clase, nombre in [(-1, "no aprobó (−1)"), (1, "aprobó (+1)")]:
        eje.scatter(*X[y == clase].T, color=COLORES[clase], s=s, edgecolor="black",
                    linewidth=0.5, label=nombre, zorder=3)


def dibujar_recta(eje, w, b, desplazamiento=0.0, lim=LIM, **kwargs):
    """Recta w·x + b = desplazamiento (se asume w[1] != 0)."""
    x1 = np.array(lim)
    eje.plot(x1, (desplazamiento - b - w[0] * x1) / w[1], **kwargs)

El perceptrón (Rosenblatt, 1958) es el clasificador lineal más antiguo: recorre los datos
y corrige la recta cada vez que se equivoca, hasta no equivocarse más. Si las clases son
separables, siempre termina — pero **dónde** termina depende del orden en que vio los
datos. Tres órdenes distintos, tres rectas, las tres con error cero:

In [ ]:
perceptrones = {}
for semilla in [0, 1, 3]:
    p = Perceptron(random_state=semilla).fit(X, y)
    perceptrones[f"perceptrón {semilla}"] = (p.coef_[0], p.intercept_[0])
    print(f"perceptrón {semilla}: errores en entrenamiento = {np.sum(p.predict(X) != y)}")

fig, eje = plt.subplots(figsize=(5.8, 5.8))
for (nombre, (w, b)), color in zip(perceptrones.items(), ["tab:green", "tab:orange", "tab:purple"]):
    dibujar_recta(eje, w, b, color=color, label=nombre)
dibujar_puntos(eje, X, y)
eje.legend(loc="upper left", fontsize=9)
preparar_eje(eje, "Tres rectas que separan perfectamente")
plt.show()

Las tres son "perfectas" en entrenamiento, pero no se ven igual de confiables: algunas
pasan rozando a un estudiante. Con un estudiante nuevo parecido a ese, cualquier pequeña
diferencia lo pondría del otro lado. Para elegir entre ellas hace falta medir **cuánto
espacio deja cada recta** — y para eso, un poco de geometría.

## 2. La geometría de una recta

Una recta del plano se escribe $f(\mathbf{x}) = \mathbf{w}^\top\mathbf{x} + b = 0$. Dos
hechos la hacen fácil de leer:

- $\mathbf{w}$ es **perpendicular** a la recta: si $\mathbf{x}_A$ y $\mathbf{x}_B$ están
  sobre ella, $\mathbf{w}^\top(\mathbf{x}_A - \mathbf{x}_B) = 0$. Apunta hacia el lado
  donde $f > 0$ (los que la recta clasifica como $+1$).
- La **distancia con signo** de un punto a la recta es
  $f(\mathbf{x}) / \lVert\mathbf{w}\rVert$: el valor de $f$ mide la distancia, en unidades
  de $\lVert\mathbf{w}\rVert$.

Multiplicando por la etiqueta, $y_i f(\mathbf{x}_i) / \lVert\mathbf{w}\rVert$ es positivo
si el punto está del lado correcto, y su valor es la distancia a la recta. El **margen**
de una recta es la menor de esas distancias: el punto que más cerca pasa. Alrededor de la
recta queda una **calle** vacía de ancho $2 \times$ margen.

In [ ]:
def margen(w, b, X, y):
    """Distancia (con signo) del punto peor ubicado a la recta w·x + b = 0."""
    return np.min(y * (X @ w + b)) / np.linalg.norm(w)


def dibujar_calle(eje, w, b, X, y, color, etiqueta):
    m = margen(w, b, X, y)
    norma = np.linalg.norm(w)
    # La calle va de f = -m·|w| a f = +m·|w|
    x1 = np.linspace(*LIM, 2)
    borde_inf = (-m * norma - b - w[0] * x1) / w[1]
    borde_sup = (m * norma - b - w[0] * x1) / w[1]
    eje.fill_between(x1, borde_inf, borde_sup, color=color, alpha=0.2, zorder=1)
    dibujar_recta(eje, w, b, color=color, linewidth=2, label=etiqueta)
    # El punto que define el margen, con su distancia perpendicular a la recta
    i = np.argmin(y * (X @ w + b))
    pie = X[i] - (X[i] @ w + b) / norma**2 * w
    eje.plot(*np.c_[X[i], pie], color="black", linewidth=1.2, zorder=4)
    return m


fig, ejes = plt.subplots(1, 3, figsize=(16, 5.5))
for eje, ((nombre, (w, b)), color) in zip(ejes, zip(perceptrones.items(),
                                                     ["tab:green", "tab:orange", "tab:purple"])):
    dibujar_puntos(eje, X, y)
    m = dibujar_calle(eje, w, b, X, y, color, nombre)
    # La flecha de w, desde un punto de la recta
    origen = -b * w / np.linalg.norm(w) ** 2
    eje.annotate("", xy=origen + 0.8 * w / np.linalg.norm(w), xytext=origen,
                 arrowprops=dict(arrowstyle="->", color=color, linewidth=2))
    eje.text(*(origen + 0.9 * w / np.linalg.norm(w)), r"$\mathbf{w}$", color=color, fontsize=13)
    preparar_eje(eje, f"{nombre}: margen = {m:.3f}")
plt.tight_layout()
plt.show()

Cada recta con su calle, la flecha de $\mathbf{w}$ y, en negro, la distancia
perpendicular al punto que la limita. Las tres separan, pero las calles son muy
distintas: la del perceptrón 0 es diez veces más ancha que la del 3, y las de los
perceptrones 1 y 3 son tan angostas que apenas se ven — pasan rozando a un estudiante.

### Una normalización que simplifica todo

$\mathbf{w}$ y $b$ no son únicos: $(2\mathbf{w}, 2b)$ describe la misma recta. Podemos
entonces escalarlos de manera que el punto más cercano tenga $y_i f(\mathbf{x}_i) = 1$
exactamente. Con esa elección los bordes de la calle son las rectas $f = +1$ y $f = -1$, y
el margen es simplemente

$$
\text{margen} = \frac{1}{\lVert\mathbf{w}\rVert}, \qquad
\text{ancho de la calle} = \frac{2}{\lVert\mathbf{w}\rVert}.
$$

Maximizar la calle es **minimizar $\lVert\mathbf{w}\rVert$** sujeto a que todos los
puntos queden fuera de ella: el problema de `02-knn-y-svm.md`,

$$
\min_{\mathbf{w}, b} \; \tfrac{1}{2}\lVert\mathbf{w}\rVert^2
\quad \text{sujeto a} \quad y_i(\mathbf{w}^\top\mathbf{x}_i + b) \geq 1 \;\; \forall i.
$$

## 3. La calle más ancha, a mano

Son 3 incógnitas ($w_1, w_2, b$), una función cuadrática y 20 restricciones lineales:
un problema pequeño que `scipy.optimize.minimize` resuelve directamente.

In [ ]:
def svm_margen_duro(X, y):
    objetivo = lambda v: 0.5 * np.sum(v[:-1] ** 2)
    restricciones = {"type": "ineq", "fun": lambda v: y * (X @ v[:-1] + v[-1]) - 1}  # >= 0
    sol = minimize(objetivo, np.zeros(X.shape[1] + 1), method="SLSQP",
                   constraints=[restricciones], options={"ftol": 1e-12})
    return sol.x[:-1], sol.x[-1]


w_svm, b_svm = svm_margen_duro(X, y)
svc_duro = SVC(kernel="linear", C=1e6, tol=1e-8).fit(X, y)  # C enorme = no tolerar invasiones

print(f"A mano:       w = {w_svm.round(4)}, b = {b_svm:.4f}")
print(f"scikit-learn: w = {svc_duro.coef_[0].round(4)}, b = {svc_duro.intercept_[0]:.4f}")
print(f"Margen = 1/|w| = {1 / np.linalg.norm(w_svm):.4f}")

valores_f = y * (X @ w_svm + b_svm)
soporte = np.where(np.isclose(valores_f, 1, atol=1e-6))[0]
print(f"\ny·f(x) de cada estudiante:\n{np.sort(valores_f).round(3)}")
print(f"Vectores de soporte (y·f = 1): {sorted(soporte.tolist())}   "
      f"scikit-learn: {sorted(svc_duro.support_.tolist())}")

In [ ]:
fig, eje = plt.subplots(figsize=(6.5, 6.5))
dibujar_puntos(eje, X, y)
x1 = np.array(LIM)
eje.fill_between(x1, (-1 - b_svm - w_svm[0] * x1) / w_svm[1],
                 (1 - b_svm - w_svm[0] * x1) / w_svm[1], color="gold", alpha=0.3, zorder=1)
dibujar_recta(eje, w_svm, b_svm, color="black", linewidth=2, label="SVM: f = 0")
dibujar_recta(eje, w_svm, b_svm, 1, color="black", linestyle="--", linewidth=1,
              label="bordes: f = ±1")
dibujar_recta(eje, w_svm, b_svm, -1, color="black", linestyle="--", linewidth=1)
eje.scatter(*X[soporte].T, s=260, facecolor="none", edgecolor="black", linewidth=1.8,
            zorder=4, label="vectores de soporte")
eje.legend(loc="upper left", fontsize=9)
preparar_eje(eje, f"Margen máximo = {1 / np.linalg.norm(w_svm):.3f}")
plt.show()

La calle de la SVM es más ancha que la de cualquiera de los perceptrones, y la tocan
exactamente **tres estudiantes**: los vectores de soporte, los únicos con
$y_i f(\mathbf{x}_i) = 1$. Todos los demás tienen $y_i f(\mathbf{x}_i) > 1$: están fuera
de la calle y su restricción no está "activa".

> En unidades estandarizadas la calle mide algo más de 0.2 desviaciones estándar: en esta
> muestra los aprobados y no aprobados casi se tocan. La SVM no promete una calle ancha;
> promete la más ancha posible.

### Solo los vectores de soporte importan

Si la solución depende solo de las restricciones activas, quitar a cualquier otro
estudiante no debería cambiar nada. Quitar a un vector de soporte, en cambio, sí.
Comparemos con la regresión logística (con su regularización por defecto), donde
**todos** los puntos aportan al gradiente.

In [ ]:
no_soporte = [i for i in range(len(y)) if i not in soporte]
quitar = {
    "sin un estudiante lejano": no_soporte[int(np.argmax(valores_f[no_soporte]))],
    "sin un vector de soporte": soporte[0],
}

logistica = LogisticRegression().fit(X, y)
filas = []
fig, ejes = plt.subplots(1, 2, figsize=(13, 6))
for eje, (nombre, i) in zip(ejes, quitar.items()):
    resto = np.delete(np.arange(len(y)), i)
    w2, b2 = svm_margen_duro(X[resto], y[resto])
    log2 = LogisticRegression().fit(X[resto], y[resto])
    filas.append({
        "caso": nombre,
        "cambio en w de la SVM": np.linalg.norm(w2 - w_svm),
        "cambio en w de la logística": np.linalg.norm(log2.coef_[0] - logistica.coef_[0]),
    })
    dibujar_recta(eje, w_svm, b_svm, color="black", linewidth=2, label="SVM con los 20")
    dibujar_recta(eje, w2, b2, color="tab:orange", linewidth=2, linestyle="--",
                  label="SVM sin el marcado")
    dibujar_puntos(eje, X[resto], y[resto])
    eje.scatter(*X[i], marker="x", s=200, color="black", linewidth=3, zorder=5,
                label="estudiante quitado")
    eje.legend(loc="upper left", fontsize=9)
    preparar_eje(eje, nombre)
plt.tight_layout()
plt.show()

pd.DataFrame(filas).set_index("caso").round(4)

Sin el estudiante lejano, la SVM es **idéntica** (cambio del orden de la tolerancia
numérica); la logística sí se mueve, porque ese estudiante también tiraba de ella. Sin un
vector de soporte, la calle de la SVM se reacomoda a los soportes que quedan.

Esto tiene dos caras. La buena: la SVM es inmune a los puntos fáciles, y para predecir
solo necesita guardar los vectores de soporte, no los 20 datos (KNN los guarda todos). La
mala: con hard margin, **un solo** punto mal etiquetado cerca de la frontera la mueve
entera. Y con datos reales, que nunca son separables, el hard margin ni siquiera tiene
solución.

## 4. Soft margin: la calle se puede invadir, pagando

Tomemos 60 estudiantes: ya no hay ninguna recta que los separe. El soft margin permite
que cada punto invada la calle una cantidad $\xi_i \geq 0$ y cobra $C$ por unidad
invadida:

$$
\min_{\mathbf{w}, b, \boldsymbol{\xi}} \; \tfrac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum_i \xi_i
\quad \text{sujeto a} \quad y_i(\mathbf{w}^\top\mathbf{x}_i + b) \geq 1 - \xi_i, \;\;
\xi_i \geq 0.
$$

Son $3 + 60$ incógnitas; `scipy` lo resuelve igual de bien.

In [ ]:
muestra60 = datos.sample(n=60, random_state=SEMILLA)
X60_crudo = muestra60[VARIABLES].to_numpy()
X60 = (X60_crudo - X60_crudo.mean(axis=0)) / X60_crudo.std(axis=0)
y60 = 2 * muestra60["aprobo"].to_numpy() - 1


def svm_margen_blando(X, y, C):
    n, p = X.shape
    objetivo = lambda v: 0.5 * np.sum(v[:p] ** 2) + C * np.sum(v[p + 1:])
    restricciones = {"type": "ineq", "fun": lambda v: y * (X @ v[:p] + v[p]) - 1 + v[p + 1:]}
    limites = [(None, None)] * (p + 1) + [(0, None)] * n  # xi >= 0
    sol = minimize(objetivo, np.zeros(p + 1 + n), method="SLSQP", constraints=[restricciones],
                   bounds=limites, options={"ftol": 1e-12, "maxiter": 1000})
    return sol.x[:p], sol.x[p], sol.x[p + 1:]


C = 1.0
w_b, b_b, xi = svm_margen_blando(X60, y60, C)
svc_blando = SVC(kernel="linear", C=C, tol=1e-8).fit(X60, y60)
print(f"A mano:       w = {w_b.round(4)}, b = {b_b:.4f}")
print(f"scikit-learn: w = {svc_blando.coef_[0].round(4)}, b = {svc_blando.intercept_[0]:.4f}")

Con el soft margin hay **tres clases de puntos**, según cuánto invaden:

| Holgura | Dónde está el punto | ¿Es vector de soporte? |
|---|---|---|
| $\xi_i = 0$ y $y_i f > 1$ | Fuera de la calle, del lado correcto | No |
| $\xi_i = 0$ y $y_i f = 1$ | Justo en el borde | Sí |
| $0 < \xi_i \leq 1$ | Dentro de la calle, todavía del lado correcto | Sí |
| $\xi_i > 1$ | Del lado equivocado de la recta: mal clasificado | Sí |

In [ ]:
f60 = X60 @ w_b + b_b
tipo = np.select([xi > 1 + 1e-6, xi > 1e-6, np.isclose(y60 * f60, 1, atol=1e-4)],
                 ["mal clasificado (ξ > 1)", "dentro de la calle (0 < ξ ≤ 1)", "en el borde (ξ = 0)"],
                 default="fuera de la calle")
print(pd.Series(tipo).value_counts().to_string())
print(f"\nVectores de soporte: {np.sum(tipo != 'fuera de la calle')} a mano, "
      f"{len(svc_blando.support_)} en scikit-learn")


def panel_blando(eje, w, b, X, y, titulo, marcar_soporte=None):
    x1 = np.array(LIM)
    eje.fill_between(x1, (-1 - b - w[0] * x1) / w[1], (1 - b - w[0] * x1) / w[1],
                     color="gold", alpha=0.3, zorder=1)
    dibujar_recta(eje, w, b, color="black", linewidth=2)
    dibujar_recta(eje, w, b, 1, color="black", linestyle="--", linewidth=1)
    dibujar_recta(eje, w, b, -1, color="black", linestyle="--", linewidth=1)
    dibujar_puntos(eje, X, y, s=30)
    if marcar_soporte is not None:
        eje.scatter(*X[marcar_soporte].T, s=150, facecolor="none", edgecolor="black",
                    linewidth=1.2, zorder=4)
    preparar_eje(eje, titulo)


fig, eje = plt.subplots(figsize=(6.5, 6.5))
panel_blando(eje, w_b, b_b, X60, y60, f"Soft margin, C = {C}: {np.sum(xi > 1e-6)} invaden la calle",
             marcar_soporte=tipo != "fuera de la calle")
for i in np.where(xi > 1e-6)[0]:  # la invasión de cada punto, hasta su borde
    pie = X60[i] - (y60[i] * f60[i] - 1) * y60[i] * w_b / np.linalg.norm(w_b) ** 2
    eje.plot(*np.c_[X60[i], pie], color=COLORES[y60[i]], linewidth=1.5, zorder=2)
plt.show()

Cada segmento de color une a un estudiante que invade con el borde de **su** lado de la
calle: su largo es $\xi_i / \lVert\mathbf{w}\rVert$. Es la misma idea que los residuales
del módulo 3: lo que el modelo no logra explicar, medido como una distancia.

### La pérdida bisagra

En el óptimo cada holgura vale exactamente lo que le falta al punto para llegar a su
borde: $\xi_i = \max(0,\, 1 - y_i f(\mathbf{x}_i))$. Sustituyendo, el problema con
restricciones se vuelve uno **sin** restricciones,

$$
\min_{\mathbf{w}, b} \; \tfrac{1}{2}\lVert\mathbf{w}\rVert^2 +
C\sum_i \max\bigl(0,\, 1 - y_i f(\mathbf{x}_i)\bigr),
$$

que tiene la misma forma que la logística regularizada del notebook 01: un término de
penalización más una suma de pérdidas, una por punto. Solo cambia la pérdida. Ambas
dependen de una sola cantidad, $m_i = y_i f(\mathbf{x}_i)$, el "margen funcional" del
punto:

In [ ]:
m = np.linspace(-2.5, 3.5, 400)
fig, eje = plt.subplots(figsize=(8, 4.5))
eje.plot(m, (m <= 0).astype(float), color="gray", linestyle=":", linewidth=2,
         label="0-1 (lo que de verdad cuenta: ¿acertó?)")
eje.plot(m, np.maximum(0, 1 - m), color="black", linewidth=2, label="bisagra (SVM)")
eje.plot(m, np.log2(1 + np.exp(-m)), color="tab:green", linewidth=2,
         label="entropía cruzada (logística, en bits)")
eje.axvline(0, color="gray", linewidth=0.8)
eje.axvline(1, color="gray", linewidth=0.8, linestyle="--")
eje.text(1.05, 2.6, "borde de\nla calle", fontsize=9)
eje.text(-2.4, 0.25, "lado equivocado", fontsize=9)
eje.text(1.6, 0.25, "fuera de la calle", fontsize=9)
eje.set_xlabel(r"$m = y\,f(\mathbf{x})$")
eje.set_ylabel("pérdida")
eje.set_ylim(-0.1, 3.5)
eje.legend()
plt.tight_layout()
plt.show()

Las dos son versiones convexas —optimizables— de la pérdida 0-1, y se parecen mucho:
lineales a la izquierda, casi cero a la derecha. La diferencia está en $m > 1$: la bisagra
es **exactamente cero**, la entropía cruzada nunca. Por eso los puntos fuera de la calle
no aportan nada a la SVM (no son vectores de soporte) y sí, un poco, a la logística. Es
la versión en una gráfica de la sección 3.

### $C$ es el precio de invadir

$C$ grande: invadir es caro, la calle se estrecha para dejar fuera a todos los que pueda.
$C$ pequeño: invadir es barato, la calle se ensancha y casi todos los puntos terminan
siendo vectores de soporte. Es la regularización al revés: $C$ juega el papel de
$1/\lambda$, como en `LogisticRegression`.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(17, 5.8))
for eje, C_ in zip(ejes, [0.1, 1, 100]):
    s = SVC(kernel="linear", C=C_, tol=1e-8).fit(X60, y60)
    panel_blando(eje, s.coef_[0], s.intercept_[0], X60, y60,
                 f"C = {C_}: calle de {2 / np.linalg.norm(s.coef_[0]):.2f}, "
                 f"{len(s.support_)} soportes, acierto {s.score(X60, y60):.2f}",
                 marcar_soporte=s.support_)
plt.tight_layout()
plt.show()

Con $C = 0.1$ la calle mide más de cuatro desviaciones estándar: la recta $f = 0$ queda
arrinconada abajo a la izquierda, fuera del alcance de los datos, y la SVM predice
"aprueba" para los 60 — acierto 0.78, el de la clase mayoritaria. Es el extremo sesgado,
como Ridge con $\lambda$ enorme. Con $C = 1$ la calle se estrecha a 1.28 y la recta ya
corta la zona de mezcla. Con $C = 100$ la calle mide 0.98 y la definen 20 puntos; a
partir de $C pprox 10$ ya casi no cambia, porque los puntos que siguen invadiendo no
pueden sacarse de la calle con ninguna recta. Cuál generaliza mejor no se ve en la
figura: se elige con validación cruzada, como el $k$ de KNN.

## 5. El truco del kernel

Hasta aquí la frontera es una recta. ¿Qué hacer cuando ninguna recta sirve?

### En una dimensión: levantar los datos

Un ejemplo de juguete: la clase $+1$ ocupa el centro de la recta real y la $-1$ los dos
extremos. Ningún umbral único los separa. Pero si a cada punto le añadimos una segunda
coordenada, $x^2$, los del centro quedan abajo y los de los extremos arriba, y una recta
**horizontal** los separa.

In [ ]:
x_1d = np.sort(rng.uniform(-3, 3, 30))
y_1d = np.where(np.abs(x_1d) < 1.2, 1, -1)

svc_1d = SVC(kernel="linear", C=1e6, tol=1e-8).fit(np.c_[x_1d, x_1d**2], y_1d)
w_1d, b_1d = svc_1d.coef_[0], svc_1d.intercept_[0]

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
for clase in (-1, 1):
    ejes[0].scatter(x_1d[y_1d == clase], np.zeros(np.sum(y_1d == clase)), color=COLORES[clase],
                    s=50, edgecolor="black", linewidth=0.5)
ejes[0].set_yticks([])
ejes[0].set_xlabel("x")
ejes[0].set_title("Una dimensión: ningún umbral los separa")

xs = np.linspace(-3.2, 3.2, 200)
ejes[1].plot(xs, xs**2, color="lightgray", zorder=0)
for clase in (-1, 1):
    ejes[1].scatter(x_1d[y_1d == clase], x_1d[y_1d == clase] ** 2, color=COLORES[clase], s=50,
                    edgecolor="black", linewidth=0.5, zorder=3)
dibujar_recta(ejes[1], w_1d, b_1d, lim=(-3.2, 3.2), color="black", linewidth=2)
ejes[1].set_ylim(-0.5, 9.5)
ejes[1].set_xlabel("x")
ejes[1].set_ylabel(r"$x^2$")
ejes[1].set_title(r"Levantados a $(x, x^2)$: una recta los separa")

# La recta del plano (x, x^2), devuelta a la recta real: dos umbrales
raices = np.roots([w_1d[1], w_1d[0], b_1d]).real
for r in raices:
    ejes[0].axvline(r, color="black", linewidth=2)
    ejes[1].axvline(r, color="gray", linestyle=":", linewidth=1)
plt.tight_layout()
plt.show()

print(f"Umbrales en la recta original: {np.sort(raices).round(3)}")

Una recta en el espacio levantado es, de vuelta en el original, **dos** umbrales: una
frontera no lineal. Es lo mismo que hacía la regresión polinómica del módulo 3 al añadir
$x^2$ como columna.

### En dos dimensiones: un plano que corta un paraboloide

Con dos clases en anillos concéntricos pasa lo mismo una dimensión más arriba. Añadiendo
$z = x_1^2 + x_2^2$ (el cuadrado de la distancia al origen), el anillo interior queda
abajo y el exterior arriba, sobre un **paraboloide**, y un **plano** horizontal los
separa.

In [ ]:
X_anillos, y_anillos = make_circles(n_samples=200, factor=0.45, noise=0.08, random_state=SEMILLA)
y_anillos = 2 * y_anillos - 1  # +1 = anillo interior
z_anillos = np.sum(X_anillos**2, axis=1)

svc_3d = SVC(kernel="linear", C=1e6, tol=1e-8).fit(np.c_[X_anillos, z_anillos], y_anillos)
w3, b3 = svc_3d.coef_[0], svc_3d.intercept_[0]

fig = plt.figure(figsize=(15, 5.5))
eje = fig.add_subplot(1, 3, 1)
for clase in (-1, 1):
    eje.scatter(*X_anillos[y_anillos == clase].T, color=COLORES[clase], s=15)
eje.set_aspect("equal")
eje.set_title("Dos anillos: ninguna recta sirve")

eje3d = fig.add_subplot(1, 3, 2, projection="3d")
# El paraboloide en coordenadas polares, hasta el radio de los datos
radio, angulo = np.meshgrid(np.linspace(0, 1.25, 20), np.linspace(0, 2 * np.pi, 40))
u, v = radio * np.cos(angulo), radio * np.sin(angulo)
eje3d.plot_surface(u, v, u**2 + v**2, color="lightgray", alpha=0.25, linewidth=0)
eje3d.plot_surface(u, v, (-b3 - w3[0] * u - w3[1] * v) / w3[2], color="gold", alpha=0.45,
                   linewidth=0)
for clase in (-1, 1):
    sel = y_anillos == clase
    eje3d.scatter(*X_anillos[sel].T, z_anillos[sel], color=COLORES[clase], s=12,
                  depthshade=False)
eje3d.set_zlim(0, 1.8)
eje3d.set_xlabel("$x_1$")
eje3d.set_ylabel("$x_2$")
eje3d.set_zlabel("$x_1^2 + x_2^2$")
eje3d.view_init(elev=12, azim=-55)
eje3d.set_title("Levantados al paraboloide: un plano los separa")

eje = fig.add_subplot(1, 3, 3)
g1, g2 = np.meshgrid(np.linspace(-1.4, 1.4, 300), np.linspace(-1.4, 1.4, 300))
f_malla = svc_3d.decision_function(np.c_[g1.ravel(), g2.ravel(), (g1**2 + g2**2).ravel()])
eje.contourf(g1, g2, (f_malla > 0).reshape(g1.shape), levels=[-0.5, 0.5, 1.5], cmap=FONDO)
eje.contour(g1, g2, f_malla.reshape(g1.shape), levels=[0], colors="black", linewidths=2)
for clase in (-1, 1):
    eje.scatter(*X_anillos[y_anillos == clase].T, color=COLORES[clase], s=15)
eje.set_aspect("equal")
eje.set_title("De vuelta en el plano: el corte es un círculo")
plt.tight_layout()
plt.show()

print(f"Plano: {w3[0]:.3f}·x1 + {w3[1]:.3f}·x2 + {w3[2]:.3f}·z + {b3:.3f} = 0")

El plano corta al paraboloide en una curva que, vista desde arriba, es un **círculo** (o
casi: el plano no es del todo horizontal porque $w_1$ y $w_2$ no son exactamente cero). La
SVM lineal en tres dimensiones es una SVM con frontera circular en dos.

### Por qué no hace falta construir el espacio

Levantar a mano funciona, pero el número de columnas explota: con $p$ variables y todos
los monomios hasta grado $d$ hay $\binom{p+d}{d}$ columnas. El truco del kernel evita
construirlas. Se apoya en un hecho de la solución de la SVM: $\mathbf{w}$ siempre es una
combinación de los vectores de soporte,

$$
\mathbf{w} = \sum_{i \in \text{soporte}} \alpha_i y_i \mathbf{x}_i
\qquad\Longrightarrow\qquad
f(\mathbf{x}) = \sum_{i \in \text{soporte}} \alpha_i y_i\, \mathbf{x}_i^\top\mathbf{x} + b,
$$

con $\alpha_i > 0$ (los coeficientes del problema *dual*, que `scikit-learn` guarda
multiplicados por $y_i$ en `dual_coef_`). Comprobémoslo con la SVM de soft margin de la
sección 4:

In [ ]:
w_desde_soportes = svc_blando.dual_coef_[0] @ svc_blando.support_vectors_
print(f"w = Σ α_i y_i x_i  = {w_desde_soportes.round(6)}")
print(f"w de scikit-learn  = {svc_blando.coef_[0].round(6)}")
print(f"α_i (= C para los que invaden, < C para los del borde): "
      f"{np.sort(np.abs(svc_blando.dual_coef_[0])).round(3)}")

Así que $f$ solo usa los datos a través de **productos punto** $\mathbf{x}_i^\top\mathbf{x}$
—y lo mismo pasa en el problema de optimización—. Si se levantan los datos con $\phi$, solo
hacen falta los productos $\phi(\mathbf{x}_i)^\top\phi(\mathbf{x})$. Y para muchos $\phi$
útiles ese producto se calcula **sin pasar por $\phi$**. Por ejemplo, con dos variables,
el kernel polinómico de grado 2

$$
K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top\mathbf{x}' + 1)^2
$$

es exactamente el producto punto en un espacio de 6 dimensiones,
$\phi(\mathbf{x}) = (1,\ \sqrt2 x_1,\ \sqrt2 x_2,\ x_1^2,\ x_2^2,\ \sqrt2 x_1x_2)$:

In [ ]:
def phi(x):
    x1, x2 = x
    r2 = np.sqrt(2)
    return np.array([1, r2 * x1, r2 * x2, x1**2, x2**2, r2 * x1 * x2])


xa, xb = rng.normal(size=2), rng.normal(size=2)
print(f"φ(a)·φ(b)  = {phi(xa) @ phi(xb):.10f}   (construye 6 columnas)")
print(f"(a·b + 1)² = {(xa @ xb + 1) ** 2:.10f}   (solo un producto punto en 2D)")

filas = []
for p in [2, 10, 100, 1000]:
    filas.append({"variables p": p, **{f"columnas, grado {d}": comb(p + d, d) for d in (2, 3, 5)}})
pd.DataFrame(filas).set_index("variables p")

El kernel cuesta un producto punto en $p$ dimensiones; el espacio que representa tiene
$\binom{p+d}{d}$: con 100 variables y grado 5, casi cien millones de columnas que nunca se
construyen.

### El kernel RBF: una montaña por vector de soporte

El kernel más usado, el RBF o gaussiano, $K(\mathbf{x}, \mathbf{x}') =
\exp(-\gamma\lVert\mathbf{x} - \mathbf{x}'\rVert^2)$, corresponde a un espacio de
dimensión **infinita**. Pero la función de decisión tiene una lectura muy concreta:

$$
f(\mathbf{x}) = \sum_{i \in \text{soporte}} \alpha_i y_i\,
e^{-\gamma\lVert\mathbf{x} - \mathbf{x}_i\rVert^2} + b
$$

es una **suma de campanas gaussianas**, una centrada en cada vector de soporte: hacia
arriba para los $+1$, hacia abajo para los $-1$, con altura $\alpha_i$. La frontera es la
curva de nivel $f = 0$ de ese paisaje. Reconstruyámosla a mano sobre los anillos.

In [ ]:
GAMMA = 1.0
svc_rbf = SVC(kernel="rbf", gamma=GAMMA, C=1.0).fit(X_anillos, y_anillos)


def f_rbf_a_mano(X_nuevos, modelo, gamma):
    d2 = np.sum((X_nuevos[:, None, :] - modelo.support_vectors_[None, :, :]) ** 2, axis=-1)
    return np.exp(-gamma * d2) @ modelo.dual_coef_[0] + modelo.intercept_[0]


puntos = np.c_[g1.ravel(), g2.ravel()]
f_mano = f_rbf_a_mano(puntos, svc_rbf, GAMMA)
print(f"Máxima diferencia con decision_function: "
      f"{np.abs(f_mano - svc_rbf.decision_function(puntos)).max():.1e}")
print(f"Vectores de soporte: {len(svc_rbf.support_)} de {len(y_anillos)}")

fig = plt.figure(figsize=(14, 6))
eje3d = fig.add_subplot(1, 2, 1, projection="3d")
eje3d.plot_surface(g1, g2, f_mano.reshape(g1.shape), cmap="RdBu", vmin=-2, vmax=2,
                   rstride=6, cstride=6, alpha=0.85, linewidth=0)
eje3d.contour(g1, g2, f_mano.reshape(g1.shape), levels=[0], colors="black", linewidths=2,
              offset=0)
eje3d.set_xlabel("$x_1$")
eje3d.set_ylabel("$x_2$")
eje3d.set_zlabel("f(x)")
eje3d.view_init(elev=30, azim=-60)
eje3d.set_title("f(x): suma de campanas sobre los vectores de soporte")

eje = fig.add_subplot(1, 2, 2)
eje.contourf(g1, g2, f_mano.reshape(g1.shape), levels=20, cmap="RdBu", vmin=-2, vmax=2)
eje.contour(g1, g2, f_mano.reshape(g1.shape), levels=[-1, 0, 1], colors="black",
            linewidths=[1, 2, 1], linestyles=["--", "-", "--"])
for clase in (-1, 1):
    eje.scatter(*X_anillos[y_anillos == clase].T, color=COLORES[clase], s=12)
eje.scatter(*svc_rbf.support_vectors_.T, s=70, facecolor="none", edgecolor="black",
            linewidth=1)
eje.set_aspect("equal")
eje.set_title("Vista desde arriba: f = 0 (frontera) y f = ±1 (calle)")
plt.tight_layout()
plt.show()

El paisaje sube en el centro (las campanas positivas del anillo interior se suman) y baja
alrededor. La frontera es su curva de nivel cero y la "calle" son las curvas $\pm 1$: el
mismo margen de la sección 3, ahora curvado. Y otra vez solo cuentan los vectores de
soporte, marcados con círculos.

### $\gamma$ es el ancho de las campanas

$\gamma$ grande hace campanas estrechas: cada vector de soporte solo influye en su
vecindad inmediata y la frontera rodea puntos sueltos, como KNN con $k$ pequeño. $\gamma$
pequeño hace campanas anchas que se solapan y la frontera se aplana hacia una recta.
Volvamos a los 400 estudiantes:

In [ ]:
X_todos = datos[VARIABLES].to_numpy()
y_todos = 2 * datos["aprobo"].to_numpy() - 1
escalador = StandardScaler().fit(X_todos)
Z_todos = escalador.transform(X_todos)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEMILLA)

GAMMAS = [0.1, 1, 10, 100]
h1, h2 = np.meshgrid(np.linspace(-3, 3.2, 250), np.linspace(-1.9, 4.2, 250))
fig, ejes = plt.subplots(1, 4, figsize=(20, 5.2))
filas = []
for eje, gamma in zip(ejes, GAMMAS):
    modelo = SVC(kernel="rbf", gamma=gamma, C=1.0).fit(Z_todos, y_todos)
    acc = cross_val_score(make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=gamma, C=1.0)),
                          X_todos, y_todos, cv=cv)
    filas.append({"γ": gamma, "vectores de soporte": len(modelo.support_),
                  "accuracy entrenamiento": modelo.score(Z_todos, y_todos),
                  "accuracy validación": acc.mean()})
    f = modelo.decision_function(np.c_[h1.ravel(), h2.ravel()]).reshape(h1.shape)
    eje.contourf(h1, h2, (f > 0).astype(int), levels=[-0.5, 0.5, 1.5], cmap=FONDO)
    eje.contour(h1, h2, f, levels=[0], colors="black", linewidths=1.2)
    for clase in (-1, 1):
        eje.scatter(*Z_todos[y_todos == clase].T, color=COLORES[clase], s=6, alpha=0.6)
    eje.set_aspect("equal")
    eje.set_xlabel("promedio (estand.)")
    eje.set_ylabel("horas (estand.)")
    eje.set_title(f"γ = {gamma}: validación {acc.mean():.3f}")
plt.tight_layout()
plt.show()

acc_log = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X_todos, y_todos,
                          cv=cv)
print(f"Referencia — regresión logística con las mismas dos variables: {acc_log.mean():.3f}")
pd.DataFrame(filas).set_index("γ").round(3)

Con $\gamma = 0.1$ la frontera es casi una recta; con $\gamma = 100$ la frontera se rompe
en islas alrededor de pequeños grupos de no aprobados, el acierto en entrenamiento se
dispara y el de validación cae.
$\gamma$ es otra perilla sesgo-varianza, y no es independiente de $C$: campanas estrechas
con $C$ grande es la combinación más flexible. Por eso `02-knn-y-svm.md` insiste en
afinarlos **juntos**, en una rejilla logarítmica.

En estos datos, la frontera de la mejor SVM es casi lineal, y una logística con las mismas
dos variables acierta prácticamente lo mismo. Lo que el kernel compra solo aparece cuando la frontera
verdadera es curva, como en los anillos. Es lo que `02-clasificacion-aplicado.ipynb`
encuentra en Wine Quality: la SVM con RBF apenas se separa de la logística.

## Resumen

| Lo que se vio | Su forma geométrica | Conecta con |
|---|---|---|
| Rectas separadoras | Infinitas; el perceptrón da una distinta según el orden de los datos | Logística (notebook 01) |
| $\mathbf{w}$, $b$ | $\mathbf{w}$ perpendicular a la recta; $f(\mathbf{x})/\lVert\mathbf{w}\rVert$ es la distancia con signo | Álgebra lineal (módulo 1) |
| Margen máximo | La calle más ancha: $2/\lVert\mathbf{w}\rVert$, minimizar $\lVert\mathbf{w}\rVert$ | `02-knn-y-svm.md`, sección 2 |
| Vectores de soporte | Los que tocan o invaden la calle; quitar cualquier otro no cambia nada | Costo de predicción; KNN guarda todos los puntos (notebook 01b) |
| Soft margin | Invadir cuesta $C\,\xi_i$; tres clases de puntos | $C = 1/\lambda$, como en `LogisticRegression` |
| Pérdida bisagra | Cero fuera de la calle; la entropía cruzada nunca es cero | Entropía cruzada (notebook 01) |
| Truco del kernel | Un plano en un espacio levantado; solo hacen falta productos punto | Regresión polinómica (módulo 3) |
| Kernel RBF | Suma de campanas sobre los soportes; $\gamma$ es su ancho | Curva en U; afinar $C$ y $\gamma$ juntos (`06-seleccion-hiperparametros.md`) |

El notebook 02 pone a competir a la logística, KNN y la SVM sobre Wine Quality, con las
métricas que este notebook dejó de lado: precisión, recall y el umbral de decisión.